In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:

df = pd.read_csv("Updated_House_Dataset.csv")

In [3]:
df

,Size_sqft,Bedrooms,Location,Age_Years,Garage,Price_USD
0,3350.0,4,Monaco,19.0,Yes,"$1,638,000"
1,1950.0,3,Dubai,NaN,Yes,"$734,200"
2,2250.0,3,Paris,21.0,Yes,"$794,200"
3,NaN,5,Singapore,27.0,Yes,"$1,210,500"
4,NaN,5,Paris,15.0,Yes,"$1,356,700"
...,...,...,...,...,...,...
118,2850.0,4,Dubai,26.0,No,"$835,800"
119,3000.0,4,Tokyo,27.0,Yes,"$1,066,400"
120,3550.0,5,Monaco,1.0,Yes,"$1,808,500"
121,3100.0,5,Paris,24.0,No,"$1,253,200"


**FEATURE LABEL SEPARATION**

In [4]:
X = df.drop("Price_USD",axis = "columns")
y = df["Price_USD"]

In [5]:
X

,Size_sqft,Bedrooms,Location,Age_Years,Garage
0,3350.0,4,Monaco,19.0,Yes
1,1950.0,3,Dubai,NaN,Yes
2,2250.0,3,Paris,21.0,Yes
3,NaN,5,Singapore,27.0,Yes
4,NaN,5,Paris,15.0,Yes
...,...,...,...,...,...
118,2850.0,4,Dubai,26.0,No
119,3000.0,4,Tokyo,27.0,Yes
120,3550.0,5,Monaco,1.0,Yes
121,3100.0,5,Paris,24.0,No


In [6]:
y

0      $1,638,000
1        $734,200
2        $794,200
3      $1,210,500
4      $1,356,700
          ...    
118      $835,800
119    $1,066,400
120    $1,808,500
121    $1,253,200
122      $334,700
Name: Price_USD, Length: 123, dtype: str

**NOW TRAIN TEST SPLITTING**

In [7]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [8]:
X_train.shape

(98, 5)

**NOW I HAVE DONE DATA SPLITTING , LETS JUST PERFORM DATA PREPROCESSING
OR FEATURE ENGINEERING**

In [9]:
#Lets first copy the splitted datasets , 
X_train_prepared  = X_train.copy()
X_test_prepared  = X_test.copy()

**HANDLING NULL / MISSING VALUES**

In [10]:
X_train_prepared.isnull().sum()

Size_sqft    1
Bedrooms     0
Location     0
Age_Years    1
Garage       0
dtype: int64

In [11]:
X_test_prepared.isnull().sum()

Size_sqft    1
Bedrooms     0
Location     0
Age_Years    0
Garage       0
dtype: int64

In [12]:
#so we have seen the null or missing values
#Mean imputation for size_sqft column

size_mean = X_train_prepared["Size_sqft"].mean()
X_train_prepared["Size_sqft"]= X_train_prepared["Size_sqft"].fillna(size_mean)
X_test_prepared["Size_sqft"] = X_test_prepared["Size_sqft"].fillna(size_mean)

In [13]:
#X_train_perpared.isnull().sum()
#X_test_prepared.isnull().sum()



In [14]:
#now do mean imputation in Age years
age_mean = X_train_prepared["Age_Years"].mean()
X_train_prepared["Age_Years"]= X_train_prepared["Age_Years"].fillna(age_mean)
X_train_prepared.isnull().sum()

Size_sqft    0
Bedrooms     0
Location     0
Age_Years    0
Garage       0
dtype: int64

**OUTLIER HANDLING**

In [15]:
#deleting a row or keeping a row is simple , we will see only the cap Function
X_train_prepared["Size_sqft"].max()

np.float64(3950.0)

In [16]:
size_upper_limit = X_train_prepared["Size_sqft"].quantile(0.95)
size_upper_limit

np.float64(3750.0)

In [17]:
#now replace the maximum values with the upper limit

X_train_prepared["Size_sqft"] = X_train_prepared["Size_sqft"].clip(upper=size_upper_limit)
X_test_prepared["Size_sqft"] = X_test_prepared["Size_sqft"].clip(upper=size_upper_limit)

#now check
X_train_prepared["Size_sqft"].max()

np.float64(3750.0)

**FEATURE SCALING**

In [18]:
X_train_prepared.head()

,Size_sqft,Bedrooms,Location,Age_Years,Garage
42,3450.0,5,Tokyo,4.0,Yes
118,2850.0,4,Dubai,26.0,No
94,3100.0,6,Tokyo,1.0,Yes
36,1900.0,4,Tokyo,7.0,No
90,3650.0,6,Tokyo,9.0,Yes


In [19]:
from sklearn.preprocessing import StandardScaler

In [20]:
#now i have numeric valued columns
numeric_columns = ["Size_sqft","Bedrooms","Age_Years"]
numeric_columns

scaler = StandardScaler()

In [21]:
X_train_prepared[numeric_columns]=scaler.fit_transform(X_train_prepared[numeric_columns])
X_test_prepared[numeric_columns] = scaler.transform(X_test_prepared[numeric_columns])

In [22]:
#now i can clearly see that my data is scaled fine
X_train_prepared

,Size_sqft,Bedrooms,Location,Age_Years,Garage
42,1.206479,0.864106,Tokyo,-1.478464,Yes
118,0.532581,0.197315,Dubai,1.118278,No
94,0.813372,1.530896,Tokyo,-1.832565,Yes
36,-0.534424,0.197315,Tokyo,-1.124363,No
90,1.431111,1.530896,Tokyo,-0.888295,Yes
...,...,...,...,...,...
106,0.701055,0.197315,London,0.055975,No
14,-1.769903,-1.136265,Manhattan,-1.596497,Yes
92,0.307948,0.864106,Manhattan,-0.416160,Yes
51,-0.422108,0.197315,Tokyo,-0.770261,No


**NOW CATEGORICAL ENCODING FOR TEXT DATA , LOCATION AND GARAGE COLUMNS**

In [23]:
#For garage column we have only yes and no so we will use binary encoding 
#let see how
X_train_prepared["Garage"] = X_train_prepared["Garage"].replace({"Yes":1, "No":0})
X_test_prepared["Garage"] = X_test_prepared["Garage"].replace({"Yes":1,"No":0})

X_train_prepared

,Size_sqft,Bedrooms,Location,Age_Years,Garage
42,1.206479,0.864106,Tokyo,-1.478464,1
118,0.532581,0.197315,Dubai,1.118278,0
94,0.813372,1.530896,Tokyo,-1.832565,1
36,-0.534424,0.197315,Tokyo,-1.124363,0
90,1.431111,1.530896,Tokyo,-0.888295,1
...,...,...,...,...,...
106,0.701055,0.197315,London,0.055975,0
14,-1.769903,-1.136265,Manhattan,-1.596497,1
92,0.307948,0.864106,Manhattan,-0.416160,1
51,-0.422108,0.197315,Tokyo,-0.770261,0


In [35]:
#now for Location Column , there are multiple values , so we will do one hot encoding
#X_train_prepared = pd.get_dummies(X_train_prepared ,columns = ["Location"],  dtype = int)
X_test_prepared = pd.get_dummies(X_test_prepared ,columns=["Location"] , dtype = int)

#X_test_prepared

In [38]:
X_train_prepared.head()

,Size_sqft,Bedrooms,Age_Years,Garage,Location_Dubai,Location_London,Location_Manhattan,Location_Monaco,Location_Paris,Location_Singapore,Location_Tokyo
42,1.206479,0.864106,-1.478464,1,0,0,0,0,0,0,1
118,0.532581,0.197315,1.118278,0,1,0,0,0,0,0,0
94,0.813372,1.530896,-1.832565,1,0,0,0,0,0,0,1
36,-0.534424,0.197315,-1.124363,0,0,0,0,0,0,0,1
90,1.431111,1.530896,-0.888295,1,0,0,0,0,0,0,1


In [39]:
X_test_prepared.head()

,Size_sqft,Bedrooms,Age_Years,Garage,Location_Dubai,Location_London,Location_Manhattan,Location_Monaco,Location_Paris,Location_Singapore,Location_Tokyo
18,-1.657587,-1.136265,-0.298126,1,0,0,0,0,0,1,0
45,-0.141317,-1.136265,-1.242396,1,0,0,0,0,0,1,0
47,0.869530,1.530896,0.292042,0,1,0,0,0,0,0,0
89,0.364106,0.197315,1.590413,1,0,1,0,0,0,0,0
4,0.005157,0.864106,-0.180093,1,0,0,0,0,1,0,0


**#SO WE ARE DONE WITH DATA PREPROCESSING**

**LETS DIVE INTO MODEL SELECTION**

In [41]:
#So i have to predict the house price so i will use the regression algorithm
#as i see how my data looks like , there are irrelevant or inordered 
#lets fix them

X_train_prepared , y_train = X_train_prepared.reset_index(drop=True),y_train.reset_index(drop=True)
X_test_prepared , y_test = X_test_prepared.reset_index(drop=True),y_testX.reset_index(drop=True)

In [43]:
X_train_prepared.head()
#now i can see indexes are ok

,Size_sqft,Bedrooms,Age_Years,Garage,Location_Dubai,Location_London,Location_Manhattan,Location_Monaco,Location_Paris,Location_Singapore,Location_Tokyo
0,1.206479,0.864106,-1.478464,1,0,0,0,0,0,0,1
1,0.532581,0.197315,1.118278,0,1,0,0,0,0,0,0
2,0.813372,1.530896,-1.832565,1,0,0,0,0,0,0,1
3,-0.534424,0.197315,-1.124363,0,0,0,0,0,0,0,1
4,1.431111,1.530896,-0.888295,1,0,0,0,0,0,0,1


In [44]:
from sklearn.linear_model import LinearRegression

In [45]:
model = LinearRegression() #object creation
model

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [58]:
#data in my y_train is string so i need to convert to float
y_train = y_train.str.replace("$","",regex = False)
y_train = y_train.str.replace(",","",regex = False)
y_train = y_train.astype(float)



In [61]:
y_train

0     1195200.0
1      835800.0
2     1329700.0
3      694500.0
4     1558500.0
        ...    
93    1206400.0
94     402300.0
95    1063400.0
96     914500.0
97    1676500.0
Name: Price_USD, Length: 98, dtype: float64

In [59]:
#training features X_train_prepared
#training labels y_train
#train the linear regression model 
model.fit(X_train_prepared,y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](11,)","[373763.81, 12854.11,-28188.81,...,-74126.31,-30749.38,-52718.95]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](11,)","['Size_sqft','Bedrooms','Age_Years',...,'Location_Paris', 'Location_Singapore','Location_Tokyo']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,9.391e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,11
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(10)
